# 01 — Data Collection

End-to-end ingest of the four primary sources used by this project. Every step is fully automated — no manual downloads.

**Sources**
- World Bank WDI / WGI (via `wbgapi`)
- WHO Global Health Observatory (OData JSON)
- Our World in Data — COVID-19 dataset (GitHub raw CSV)
- UNDP Human Development Report — composite indices time series

**Outputs**
- `data/raw/worldbank_long.csv`, `worldbank_wide.csv`
- `data/raw/who.csv`
- `data/raw/owid_covid.csv`
- `data/raw/undp_hdi.csv`
- `data/processed/panel_clean.csv` (after merge + imputation)
- `data/final/master_dataset.csv`, `data_dictionary.txt`, `data_quality_report.txt`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils.config import COUNTRIES, ISO3_LIST, WB_INDICATORS, WHO_INDICATORS, OWID_COVID_COLS, YEAR_START, YEAR_END
print(f'Countries: {len(COUNTRIES)}  Years: {YEAR_START}-{YEAR_END}')
print(f'WB indicators: {len(WB_INDICATORS)}  WHO: {len(WHO_INDICATORS)}  OWID COVID: {len(OWID_COVID_COLS)}')

## 1. Run each collector
Each module is self-contained and can be executed alone. The pipeline is idempotent — already-fetched files are skipped.

In [ ]:
from src.data import worldbank, who, owid_covid, undp
wb_df    = worldbank.collect_and_save()
who_df   = who.collect_and_save()
owid_df  = owid_covid.collect_and_save()
undp_df  = undp.collect_and_save()
print({'wb': wb_df.shape, 'who': who_df.shape, 'owid': owid_df.shape, 'undp': undp_df.shape})

## 2. Clean, impute, and feature-engineer
Country-by-country IterativeImputer (BayesianRidge); imputed values clipped to plausible ranges.

In [ ]:
from src.data import clean, features
panel, report = clean.clean()
print(f'Panel: {panel.shape}  Excluded countries: {report.excluded_countries}')
panel = features.engineer(panel)
print(f'After feature engineering: {panel.shape}')

## 3. Validate and write final outputs
Range checks, IQR outlier flags, and an IMF cross-check on GDP per capita.

In [ ]:
from src.data import build_dataset
build_dataset.main(skip_fetch=True, skip_crosscheck=False)

In [ ]:
import pandas as pd
df = pd.read_csv('../data/final/master_dataset.csv')
print('Master dataset shape:', df.shape)
print('Countries:', df['iso3'].nunique(), '| Years:', df['year'].min(), '-', df['year'].max())
df.head()